In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve
)

import lightgbm as lgb

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------
# Load Dataset
# ---------------------------------------------------

df = pd.read_csv('../data/student_loans_engineered.csv')

# ---------------------------------------------------
# Features
# ---------------------------------------------------

FEATURES = [
    'age',
    'gender_enc',
    'course_enc',
    'tier_enc',
    'gpa',
    'trend_enc',
    'loan_amt',
    'interest_rate',
    'tenure_mo',
    'moratorium_mo',
    'emi',
    'employed',
    'salary_mo',
    'emi_income',
    'months_since_disb',
    'missed_pmts_past',
    'payment_delay_avg',
    'auto_debit',
    'co_borrower',
    'family_income_mo',
    'family_support',
    'loan_burden',
    'is_early_vintage',
    'payment_score',
    'academic_risk',
    'employment_stress',
    'support_score',
    'income_adequacy'
]

FEATURES = [f for f in FEATURES if f in df.columns]

# ---------------------------------------------------
# Targets
# ---------------------------------------------------

HORIZONS = [
    'default_30d',
    'default_60d',
    'default_90d'
]

HORIZON_LABELS = [
    '30-Day',
    '60-Day',
    '90-Day'
]

results = {}

# ---------------------------------------------------
# ROC Figure
# ---------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

fig.suptitle(
    'Early Warning Models - ROC Curves',
    fontsize=13,
    fontweight='bold'
)

# ---------------------------------------------------
# Train All Models
# ---------------------------------------------------

for i, (target, label) in enumerate(
    zip(HORIZONS, HORIZON_LABELS)
):

    print(f"\nTraining {label} Model ({target})...")

    X = df[FEATURES]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    model = lgb.LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[
            lgb.early_stopping(
                30,
                verbose=False
            )
        ]
    )

    # ---------------------------------------------------
    # Predictions
    # ---------------------------------------------------

    y_proba = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)

    auc_pr = average_precision_score(
        y_test,
        y_proba
    )

    print(f"AUC-ROC: {auc:.4f}")
    print(f"AUC-PR : {auc_pr:.4f}")

    # ---------------------------------------------------
    # ROC Curve
    # ---------------------------------------------------

    fpr, tpr, _ = roc_curve(
        y_test,
        y_proba
    )

    colors_r = [
        '#16A34A',
        '#EAB308',
        '#EF4444'
    ]

    axes[i].plot(
        fpr,
        tpr,
        color=colors_r[i],
        lw=2.5,
        label=f'{label} (AUC={auc:.3f})'
    )

    axes[i].plot(
        [0, 1],
        [0, 1],
        '--',
        color='gray',
        alpha=0.5
    )

    axes[i].set_title(
        f'{label} Default Model',
        fontweight='bold'
    )

    axes[i].legend()

    # ---------------------------------------------------
    # Save Metrics
    # ---------------------------------------------------

    results[target] = {
        'auc_roc': round(auc, 4),
        'auc_pr': round(auc_pr, 4),
        'default_rate': round(float(y.mean()), 4),
        'label': label
    }

    # ---------------------------------------------------
    # Save Model
    # ---------------------------------------------------

    joblib.dump(
        model,
        f'../models/model_{target}.pkl'
    )

    print(
        f"Saved: models/model_{target}.pkl"
    )

# ---------------------------------------------------
# Save ROC Plot
# ---------------------------------------------------

plt.tight_layout()

plt.savefig(
    '../plots/roc_curves_all_horizons.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

# ---------------------------------------------------
# Save Metrics JSON
# ---------------------------------------------------

json.dump(
    {
        'features': FEATURES,
        'horizons': results
    },
    open('../models/model_metrics.json', 'w'),
    indent=2
)

# ---------------------------------------------------
# Final Summary
# ---------------------------------------------------

print("\nSUMMARY TABLE:")

print(
    f"{'Horizon':12} | "
    f"{'Default%':8} | "
    f"{'AUC-ROC':8} | "
    f"{'AUC-PR':8}"
)

print("-" * 50)

for t, label in zip(HORIZONS, HORIZON_LABELS):

    r = results[t]

    print(
        f"{label:12} | "
        f"{r['default_rate']*100:7.1f}% | "
        f"{r['auc_roc']:8.4f} | "
        f"{r['auc_pr']:8.4f}"
    )


Training 30-Day Model (default_30d)...
AUC-ROC: 0.7326
AUC-PR : 0.2066
Saved: models/model_default_30d.pkl

Training 60-Day Model (default_60d)...
AUC-ROC: 0.7601
AUC-PR : 0.4149
Saved: models/model_default_60d.pkl

Training 90-Day Model (default_90d)...
AUC-ROC: 0.7900
AUC-PR : 0.6008
Saved: models/model_default_90d.pkl

SUMMARY TABLE:
Horizon      | Default% | AUC-ROC  | AUC-PR  
--------------------------------------------------
30-Day       |     9.5% |   0.7326 |   0.2066
60-Day       |    20.3% |   0.7601 |   0.4149
90-Day       |    31.3% |   0.7900 |   0.6008
